## **2. Workflow Type 2 : LLM-Based Sequential Workflow**

A sequential flow powered by an LLM (like GPT), where each step uses AI to process or transform data.

👉 Still step-by-step, but smarter

👉 Each step involves reasoning, generation, or understanding

**Example:
Input → LLM summarizes → LLM analyzes → LLM generates answer**

## ***Step 1 : Understand the Problem Statements***

In [1]:
#AI story generation

In [2]:
!pip install langchain langgraph langchain_google_genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 16.4 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28


In [3]:
import os

from google.colab import userdata
gem = userdata.get('gemini')

os.environ["GOOGLE_API_KEY"]=gem

## ***Step 2 : Create a state***
- Using PyDantic/TypeDict : Mostly we are using Pydantic: Create our own schema using them (Template).

- They define the structure, type, and rules of your state (shared memory) so every node in the graph works consistently and safely.

In [4]:
from pydantic import BaseModel,Field

In [6]:
# Create our state schema

class Data(BaseModel):
  topic:str
  story:str| None = Field(default=None)

In [8]:
x = Data(topic = "comedy")

# It will create pydantic object

In [9]:
x.topic

'comedy'

In [10]:
x.topic="Emotional"

In [11]:
x

Data(topic='Emotional', story=None)

## ***Step 3: Create a State Graph Object***

- With the help of state graph object we can able create entire state graph.
- This object will tell how to add nodws, how to add edges so the final graph will be designed

In [15]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate,HumanMessagePromptTemplate

In [16]:
# Creating our state graph object
graph = StateGraph(state_schema=Data)

## ***Step 4: Create Graph using Nodes(Steps) and Edges(connection)***

In [18]:
# Create ? Design the node

def llm_workflow(state:Data):
  cpt = ChatPromptTemplate.from_messages([SystemMessagePromptTemplate.from_template("You are having a experience in generating story"),
                                          HumanMessagePromptTemplate.from_template("Based on this topic:{topic} generate me a 3 lines of story")])

  model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
  app = cpt | model

  answer = app.invoke({"topic":state.topic})
  state.story = answer.content
  return state

In [19]:
# Add the nodes

graph.add_node("llms",llm_workflow)

In [20]:
# Add the edges
graph.add_edge(START,"llms")
graph.add_edge("llms",END)

## ***Step 5 : Compile the graph***

In [21]:
state_graph = graph.compile()

In [23]:
state_graph.invoke({"topic":"adventure"})

{'topic': 'adventure',
 'story': 'Elara, map clutched tight, plunged into the whispering, ancient forest, seeking the legendary Sunstone. She scaled treacherous cliffs, her heart pounding with the thrill of discovery, leaving civilization far behind. Ahead, a glimmer promised either fortune or the greatest challenge of her life.'}

In [25]:
state_graph.invoke(Data(topic="adventure"))

{'topic': 'adventure',
 'story': 'Elara, map clutched tight, ventured into the Whispering Woods, seeking the forgotten Sunstone.\nThrough ancient ruins and across treacherous ravines, she outwitted forest guardians and solved riddles of old.\nFinally, deep within a hidden grotto, the stone pulsed with an ethereal glow, promising power beyond her wildest dreams.'}